# SIC8 | 03 — Historical Silver Processing

**Scope:** Build a reproducible, source-aware Silver candidate from the full IBM historical snapshot. Original files stay unchanged.

**Outputs:** validated historical transactions, isolated ground-truth labels, and rejected-record candidates. `ENABLE_WRITE=False` by default.

**Channel rule (team assumption):** Online → ECOMMERCE; Swipe → POS; Chip → ATM. These are **project-assigned channels**, not source-proven business channels. Keep their provenance in Silver.

## 01 — Setup
Use the existing Unity Catalog Volume. Run the complete historical snapshot for stable duplicate-occurrence IDs; do not write results until all checks pass.

In [0]:
from pyspark.sql import functions as F, types as T, Window
from pyspark.sql.functions import broadcast
import time

CATALOG = "workspace"
INPUT_SCHEMA = "sic8_sources"
SILVER_SCHEMA = f"{CATALOG}.sic8_silver"  # Proposed managed schema
RAW = f"/Volumes/{CATALOG}/{INPUT_SCHEMA}/historical_raw"

FILES = {
    "transactions": f"{RAW}/credit_card_transactions-ibm_v2.csv",
    "users": f"{RAW}/sd254_users.csv",
    "cards": f"{RAW}/sd254_cards.csv",
    "mcc": f"{RAW}/merchant_category_codes.csv",
    "cities": f"{RAW}/city_reference.csv",
}

MODE = "FULL"            # FULL for production-sized validation; SAMPLE for code exploration only
SAMPLE_ROWS = 5000
ENABLE_WRITE = False     # Set to True only after reviewing validation results
SOURCE_VERSION = "IBM_V2"  # Project label; not an S3 object version or checksum
MAPPING_VERSION = "silver_candidate_0.3"

if MODE not in ("SAMPLE", "FULL"):
    raise ValueError("MODE must be SAMPLE or FULL")
print("Mode:", MODE, "| Target:", SILVER_SCHEMA, "| Write enabled:", ENABLE_WRITE)

Mode: FULL | Target: workspace.sic8_silver | Write enabled: False


## 02 — Load all five source files
Read each CSV separately as raw strings. Avoid reading the entire Volume as one CSV: the files have different schemas.

In [0]:
def read_csv(path):
    return (spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("enforceSchema", "false")
        .csv(path))

raw = {name: read_csv(path) for name, path in FILES.items()}
# Normalize a possible UTF-8 BOM in reference headers (e.g. mcc).
raw = {name: frame.toDF(*[col.lstrip("\ufeff") for col in frame.columns])
       for name, frame in raw.items()}

TX_COLUMNS = [
    "User", "Card", "Year", "Month", "Day", "Time", "Amount",
    "Use Chip", "Merchant Name", "Merchant City", "Merchant State",
    "Zip", "MCC", "Errors?", "Is Fraud?"
]

if raw["transactions"].columns != TX_COLUMNS:
    raise ValueError("Unexpected transaction columns; verify CSV version")

for name, frame in raw.items():
    print(f"{name}: {len(frame.columns)} columns")

source_transactions = (raw["transactions"].limit(SAMPLE_ROWS)
    if MODE == "SAMPLE" else raw["transactions"])

transactions: 15 columns
users: 18 columns
cards: 13 columns
mcc: 5 columns
cities: 7 columns


## 03 — Check reference contracts
The transaction `User` is a numeric index. The users CSV has `Person`, not a `User` key; never join transactions to a name. Cards use `(User, CARD INDEX)`.

In [0]:
required = {
    "users": ["Person", "City", "State"],
    "cards": ["User", "CARD INDEX", "Card Brand", "Card Type", "Has Chip"],
    "mcc": ["mcc", "description", "category"],
    "cities": ["city_name", "state_code", "country_code", "latitude", "longitude"],
}
for name, fields in required.items():
    missing = [field for field in fields if field not in raw[name].columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

print("Reference schemas: OK")
print("Users join needs verified file-order mapping; see Section 04.")

Reference schemas: OK
Users join needs verified file-order mapping; see Section 04.


## 04 — Build an indexed Users reference
The IBM transactions use zero-based user indices. Reconstruct the small users CSV line order, parse quoted CSV values, and compare counts with the standard CSV reader. **This requires one record per physical line**; stop if that assumption fails.

In [0]:

# Read the original Users file while preserving row order
users_binary = spark.read.format("binaryFile").load(FILES["users"])

users_lines = (
    users_binary
    .select(
        F.posexplode(
            F.split(
                F.decode(F.col("content"), "UTF-8"),
                r"\r?\n"
            )
        ).alias("line_no", "csv_line")
    )
    .filter(
        (F.col("line_no") > 0) &
        (F.trim(F.col("csv_line")) != "")
    )
)

# Convert StructType into a DDL schema string
users_ddl = raw["users"].schema.toDDL()

users_indexed = (
    users_lines
    .select(
        (F.col("line_no") - 1)
            .cast("string")
            .alias("_user_key"),

        F.from_csv(
            F.col("csv_line"),
            users_ddl
        ).alias("user_record")
    )
    .select("_user_key", "user_record.*")
)

# Validate reconstructed records
raw_user_count = raw["users"].count()
indexed_user_count = users_indexed.count()

if raw_user_count != indexed_user_count:
    raise ValueError("User row count mismatch")

if users_indexed.filter(F.col("Person").isNull()).limit(1).count():
    raise ValueError("Users CSV parsing failed")

if users_indexed.filter(F.col("_user_key").isNull()).limit(1).count():
    raise ValueError("Missing user indices")

users_ref = users_indexed.select(
    "_user_key",
    F.col("City").alias("customer_city"),
    F.col("State").alias("customer_region"),
    F.lit(True).alias("user_found")
)

print("Original users:", raw_user_count)
print("Indexed users:", indexed_user_count)

display(
    users_indexed.select(
        "_user_key", "City", "State"
    ).limit(5)
)


Original users: 2000
Indexed users: 2000


_user_key,City,State
0,La Verne,CA
1,Little Neck,NY
2,West Covina,CA
3,New York,NY
4,San Francisco,CA


## 05 — Prepare safe reference tables
Keep only non-sensitive card attributes. Validate reference keys before joins: duplicate keys would multiply transaction rows.

In [0]:
ref_cards = raw["cards"].select(
    F.trim(F.col("User")).alias("_user_key"),
    F.trim(F.col("CARD INDEX")).alias("_card_key"),
    F.col("Card Brand").alias("card_brand"),
    F.col("Card Type").alias("card_type"),
    F.col("Has Chip").alias("has_chip_raw"),
    F.lit(True).alias("card_found")
)

ref_mcc = raw["mcc"].select(
    F.lpad(F.trim(F.col("mcc")), 4, "0").alias("mcc_key"),
    F.col("description").alias("mcc_description"),
    F.col("category").alias("merchant_category"),
    F.lit(True).alias("mcc_found")
)

ref_city_all = (raw["cities"]
    .filter(F.upper(F.trim(F.col("country_code"))) == "US")
    .select(
        F.upper(F.trim(F.col("city_name"))).alias("_city_key"),
        F.upper(F.trim(F.col("state_code"))).alias("_region_key"),
        F.expr("try_cast(latitude AS DOUBLE)").alias("merchant_city_centroid_lat"),
        F.expr("try_cast(longitude AS DOUBLE)").alias("merchant_city_centroid_lon")
    ))

# Only unambiguous city+state keys are allowed in a transaction join.
city_counts = ref_city_all.groupBy("_city_key", "_region_key").count()
unique_city_keys = city_counts.filter(F.col("count") == 1).drop("count")
ref_cities = ref_city_all.join(unique_city_keys, ["_city_key", "_region_key"], "inner")

checks = {
    "duplicate_card_keys": ref_cards.groupBy("_user_key", "_card_key").count()
        .filter(F.col("count") > 1).count(),
    "duplicate_mcc_keys": ref_mcc.groupBy("mcc_key").count()
        .filter(F.col("count") > 1).count(),
    "ambiguous_us_city_keys": city_counts.filter(F.col("count") > 1).count(),
}
if checks["duplicate_card_keys"] or checks["duplicate_mcc_keys"]:
    raise ValueError(f"Non-unique card/MCC reference keys: {checks}")
if users_ref.groupBy("_user_key").count().filter(F.col("count") > 1).count():
    raise ValueError("Duplicate reconstructed user indices")

print("Reference checks:", checks)
print("Excluded ambiguous US city keys from enrichment.")

Reference checks: {'duplicate_card_keys': 0, 'duplicate_mcc_keys': 0, 'ambiguous_us_city_keys': 0}
Excluded ambiguous US city keys from enrichment.


## 06 — Define the team's channel mapping
Normalize `Use Chip` without editing the raw CSV. **Chip → ATM is an explicit project assumption**; it must not be presented as an original ATM label.

In [0]:
def add_payment_mapping(df):
    method = (F.when(F.trim(F.col("Use Chip")) == "Swipe Transaction", "SWIPE")
        .when(F.trim(F.col("Use Chip")) == "Chip Transaction", "CHIP")
        .when(F.trim(F.col("Use Chip")) == "Online Transaction", "ONLINE")
        .otherwise("UNKNOWN"))
    return (df
        .withColumn("payment_method", method)
        .withColumn("channel", F.when(F.col("payment_method") == "ONLINE", "ECOMMERCE")
            .when(F.col("payment_method") == "SWIPE", "POS")
            .when(F.col("payment_method") == "CHIP", "ATM")
            .otherwise("UNKNOWN"))
        .withColumn("channel_origin", F.when(F.col("payment_method") == "ONLINE", "INFERRED_FROM_METHOD")
            .when(F.col("payment_method").isin("SWIPE", "CHIP"), "PROJECT_ASSUMPTION")
            .otherwise("UNKNOWN"))
        .withColumn("card_present", F.when(F.col("payment_method").isin("SWIPE", "CHIP"), F.lit(True))
            .when(F.col("payment_method") == "ONLINE", F.lit(False))
            .otherwise(F.lit(None).cast("boolean"))))

fixture = spark.createDataFrame([
    ("Swipe Transaction",), ("Chip Transaction",),
    ("Online Transaction",), ("other",)
], ["Use Chip"])
actual = [(x["payment_method"], x["channel"]) for x in
          add_payment_mapping(fixture).select("payment_method", "channel").collect()]
assert actual == [("SWIPE", "POS"), ("CHIP", "ATM"),
                  ("ONLINE", "ECOMMERCE"), ("UNKNOWN", "UNKNOWN")]
print("Channel rules passed 4 fixture checks.")

Channel rules passed 4 fixture checks.


## 07 — Assign a repeatable snapshot-row key
Kaggle has no original transaction ID. Hash the 14 **non-label** source fields, then number **identical-feature** occurrences within the full snapshot. Source labels are used only to order otherwise indistinguishable copies when attaching a separate ground-truth record; never use generated keys as fraud features. This preserves duplicate-looking transactions; it does not recover a real-world transaction ID.

**Important:** Keys are reproducible as a *set* for the unchanged full snapshot. The identity of two physically indistinguishable rows cannot be recovered. Never use SAMPLE-generated IDs in production.

In [0]:
non_label_fields = [F.col(name).alias(f"field_{i:02d}")
                    for i, name in enumerate(TX_COLUMNS[:-1], start=1)]

source_hashed = source_transactions.withColumn(
    "source_row_hash", F.sha2(F.to_json(F.struct(*non_label_fields)), 256))

occurrence_window = (Window.partitionBy("source_row_hash")
    .orderBy(F.coalesce(F.col("Is Fraud?"), F.lit(""))))

source_identified = (source_hashed
    .withColumn("_duplicate_occurrence", F.row_number().over(occurrence_window))
    .withColumn("source_record_id", F.sha2(F.concat(
        F.lit(SOURCE_VERSION + "|"), F.col("source_row_hash"), F.lit("|"),
        F.col("_duplicate_occurrence").cast("string")), 256))
    .withColumn("transaction_id", F.concat(F.lit("HIST_TX_"), F.col("source_record_id")))
    .withColumn("event_id", F.concat(F.lit("HIST_EVENT_"), F.col("source_record_id"))))

print("Historical keys are project-derived, not source IDs.")

Historical keys are project-derived, not source IDs.


## 08 — Parse money, time, and source identity
Use decimal values for money and `TIMESTAMP_NTZ` for historical local time. Do not claim the source timezone is UTC.

In [0]:
typed = (source_identified
    .withColumn("_user_key", F.trim(F.col("User")))
    .withColumn("_card_key", F.trim(F.col("Card")))
    .withColumn("source_merchant_id", F.trim(F.col("Merchant Name")))
    .withColumn("customer_id", F.when(
        F.col("_user_key").rlike(r"^[0-9]+$"),
        F.concat(F.lit("IBM_USER_"), F.col("_user_key"))))
    .withColumn("card_id", F.when(
        F.col("_user_key").rlike(r"^[0-9]+$") & F.col("_card_key").rlike(r"^[0-9]+$"),
        F.concat(F.lit("IBM_CARD_"), F.col("_user_key"), F.lit("_"), F.col("_card_key"))))
    .withColumn("amount", F.expr(
        "try_cast(regexp_replace(trim(`Amount`), '[$,]', '') AS DECIMAL(18,2))"))
    .withColumn("currency", F.lit("USD"))
    .withColumn("_local_timestamp_text", F.expr("""
        concat(lpad(trim(`Year`),4,'0'),'-',lpad(trim(`Month`),2,'0'),'-',
               lpad(trim(`Day`),2,'0'),' ',trim(`Time`))
    """))
    .withColumn("source_event_time_local", F.expr(
        "try_cast(_local_timestamp_text AS TIMESTAMP_NTZ)"))
    .withColumn("event_time_timezone_status", F.lit("UNKNOWN"))
    .withColumn("mcc_key", F.lpad(F.trim(F.col("MCC")), 4, "0"))
    .withColumn("merchant_city", F.col("Merchant City"))
    .withColumn("merchant_region_raw", F.col("Merchant State"))
    .withColumn("merchant_postal_code", F.regexp_replace(
        F.trim(F.col("Zip")), r"\.0$", ""))
    .withColumn("_city_key", F.upper(F.trim(F.col("Merchant City"))))
    .withColumn("_region_key", F.upper(F.trim(F.col("Merchant State"))))
    .withColumn("source_system", F.lit("IBM_KAGGLE_HISTORICAL"))
    .withColumn("source_dataset_version", F.lit(SOURCE_VERSION))
    .withColumn("mapping_version", F.lit(MAPPING_VERSION))
    .withColumn("ingestion_mode", F.lit("BATCH"))
    .withColumn("errors_raw", F.col("Errors?"))
    .withColumn("outcome_status", F.lit("UNKNOWN"))
    .withColumn("transaction_type", F.lit("UNKNOWN"))
    .withColumn("source_file", F.lit(FILES["transactions"])))

mapped = add_payment_mapping(typed)
print("Amount, local time, identity and project channels mapped.")

Amount, local time, identity and project channels mapped.


## 09 — Enrich with Users, Cards, MCC, and Cities
Use **left joins** so an absent reference does not silently remove a transaction. City centroids are approximate merchant geography, not exact transaction GPS.

In [0]:
enriched = (mapped
    .join(broadcast(users_ref), ["_user_key"], "left")
    .join(broadcast(ref_cards), ["_user_key", "_card_key"], "left")
    .join(broadcast(ref_mcc), ["mcc_key"], "left")
    .join(broadcast(ref_cities), ["_city_key", "_region_key"], "left")
    .withColumn("latitude", F.lit(None).cast("double"))
    .withColumn("longitude", F.lit(None).cast("double"))
    .withColumn("location_type", F.lit("TRANSACTION_LOCATION_NOT_RECORDED"))
    .withColumn("merchant_location_precision", F.when(
        F.col("merchant_city_centroid_lat").isNotNull() &
        F.col("merchant_city_centroid_lon").isNotNull(), "CITY_CENTROID"
    ).otherwise("UNKNOWN"))
    .withColumn("geo_eligible_for_exact_velocity", F.lit(False))
    .withColumn("atm_mcc_evidence", F.col("mcc_key") == "6011"))

print("Reference enrichments prepared; exact transaction GPS remain NULL.")

Reference enrichments prepared; exact transaction GPS remain NULL.


## 10 — Apply explicit quality rules
Reject only rows with unusable required identity, amount, time, payment method, or fraud label. Negative amounts and missing ZIP/error notes are not automatic failures.

In [0]:
checked = (enriched
    .withColumn("bad_identity", F.col("customer_id").isNull() | F.col("card_id").isNull())
    .withColumn("bad_amount", F.col("amount").isNull())
    .withColumn("bad_local_time", F.col("source_event_time_local").isNull())
    .withColumn("bad_payment_method", F.col("payment_method") == "UNKNOWN")
    .withColumn("bad_fraud_label", ~F.trim(F.coalesce(
        F.col("Is Fraud?"), F.lit(""))).isin("Yes", "No"))
    .withColumn("quality_status", F.when(
        F.col("bad_identity") | F.col("bad_amount") | F.col("bad_local_time") |
        F.col("bad_payment_method") | F.col("bad_fraud_label"), "REVIEW"
    ).otherwise("PASS"))
    .withColumn("quality_reason", F.concat_ws(" | ",
        F.when(F.col("bad_identity"), "BAD_IDENTITY"),
        F.when(F.col("bad_amount"), "INVALID_AMOUNT"),
        F.when(F.col("bad_local_time"), "INVALID_LOCAL_TIME"),
        F.when(F.col("bad_payment_method"), "UNKNOWN_METHOD"),
        F.when(F.col("bad_fraud_label"), "INVALID_SOURCE_LABEL")))
    .withColumn("ground_truth_label", F.when(F.col("Is Fraud?") == "Yes", F.lit(True))
        .when(F.col("Is Fraud?") == "No", F.lit(False))
        .otherwise(F.lit(None).cast("boolean"))))

accepted = checked.filter(F.col("quality_status") == "PASS")
rejected = checked.filter(F.col("quality_status") == "REVIEW")
print("Quality rules created. Missing card / MCC / city lookups stay visible as NULL.")

Quality rules created. Missing card / MCC / city lookups stay visible as NULL.


## 11 — Publish explicit Silver contracts in memory
Keep labels isolated from detection input. Do not include names, addresses, card numbers, or CVV in the transaction table.

In [0]:
TRANSACTION_FIELDS = [
    "source_record_id", "transaction_id", "event_id", "source_row_hash",
    "source_system", "source_dataset_version", "mapping_version", "ingestion_mode", "source_file",
    "customer_id", "card_id", "card_brand", "card_type", "has_chip_raw", "source_merchant_id",
    "source_event_time_local", "event_time_timezone_status",
    "amount", "currency", "payment_method", "channel", "channel_origin", "card_present", "transaction_type",
    "mcc_key", "mcc_description", "merchant_category", "merchant_city", "merchant_region_raw",
    "merchant_postal_code", "customer_city", "customer_region",
    "latitude", "longitude", "merchant_city_centroid_lat", "merchant_city_centroid_lon",
    "merchant_location_precision", "location_type", "geo_eligible_for_exact_velocity",
    "outcome_status", "errors_raw", "atm_mcc_evidence"
]

silver_transactions = accepted.select(*TRANSACTION_FIELDS)
silver_ground_truth = accepted.select(
    "source_record_id", "transaction_id", "event_id", "ground_truth_label",
    F.lit("IBM_SOURCE_LABEL").alias("label_source"))
silver_quarantine = rejected.select(
    "source_record_id", "quality_reason", "source_system", "source_dataset_version",
    *[F.col(field).alias(f"raw_{i:02d}") for i, field in enumerate(TX_COLUMNS, 1)]
)

assert "ground_truth_label" not in silver_transactions.columns
assert "Is Fraud?" not in silver_transactions.columns
assert "Card Number" not in silver_transactions.columns
assert "CVV" not in silver_transactions.columns
print("Contracts:", len(silver_transactions.columns), "transaction fields;",
      len(silver_ground_truth.columns), "ground-truth fields")

Contracts: 42 transaction fields; 5 ground-truth fields


## 12 — Full reconciliation and safety gates
Check all rows, not the first 5,000. Join output must preserve source row count. Review rejections, reference coverage, and ATM classification assumptions before writing.

In [0]:
start = time.perf_counter()
source_count = source_transactions.count()
summary = checked.agg(
    F.count("*").alias("rows_after_joins"),
    F.sum((F.col("quality_status") == "PASS").cast("long")).alias("accepted"),
    F.sum((F.col("quality_status") == "REVIEW").cast("long")).alias("review"),
    F.sum((F.col("amount") < 0).cast("long")).alias("negative_amounts"),
    F.sum((F.col("ground_truth_label") == True).cast("long")).alias("fraud_yes"),
    F.sum(F.col("card_found").isNull().cast("long")).alias("missing_card_reference"),
    F.sum(F.col("user_found").isNull().cast("long")).alias("missing_user_reference"),
    F.sum(F.col("mcc_found").isNull().cast("long")).alias("missing_mcc_reference"),
    F.sum((F.col("channel") == "ATM").cast("long")).alias("project_assigned_atm"),
    F.sum(((F.col("channel") == "ATM") &
            (F.col("mcc_key") != "6011")).cast("long")).alias("atm_without_mcc_6011")
).first().asDict()

if summary["rows_after_joins"] != source_count:
    raise AssertionError("Join changed the number of source transactions")
if summary["accepted"] + summary["review"] != source_count:
    raise AssertionError("Quality split lost rows")

print("Validation metrics:")
for k, v in {"source_rows": source_count, **summary}.items():
    print(f"  {k}: {v}")
print("Validation seconds:", round(time.perf_counter() - start, 2))
print("WARNING: project-assigned ATM rows are not proof of real ATM transactions.")

Validation metrics:
  source_rows: 24386900
  rows_after_joins: 24386900
  accepted: 24386900
  review: 0
  negative_amounts: 1244683
  fraud_yes: 29757
  missing_card_reference: 0
  missing_user_reference: 0
  missing_mcc_reference: 0
  project_assigned_atm: 6287598
  atm_without_mcc_6011: 6287598
Validation seconds: 34.6


## 13 — Preview safe fields
Inspect a small output without printing original names, card numbers, CVVs, or source fraud labels.

In [0]:
display(silver_transactions.select(
    "transaction_id", "customer_id", "card_id", "amount", "currency",
    "payment_method", "channel", "channel_origin", "merchant_category",
    "merchant_location_precision", "geo_eligible_for_exact_velocity"
).limit(10))
print("Ground truth remains separate:", silver_ground_truth.columns)

transaction_id,customer_id,card_id,amount,currency,payment_method,channel,channel_origin,merchant_category,merchant_location_precision,geo_eligible_for_exact_velocity
HIST_TX_44c498312be7d14d5c017f7c37d08caf94871a3d20c03a6da8295ef96a3ba185,IBM_USER_896,IBM_CARD_896_4,14.76,USD,SWIPE,POS,PROJECT_ASSUMPTION,Miscellaneous Stores,CITY_CENTROID,false
HIST_TX_89722fd412f099c5f35d3b4c3910c7a24fbfc70e9209ff4121c78631e06d5d72,IBM_USER_801,IBM_CARD_801_3,24.91,USD,ONLINE,ECOMMERCE,INFERRED_FROM_METHOD,Transportation Services,UNKNOWN,false
HIST_TX_066484cf6fdea60a569a7f8067fb409b75681e62681a289489cff89b616745d1,IBM_USER_531,IBM_CARD_531_0,33.31,USD,SWIPE,POS,PROJECT_ASSUMPTION,Miscellaneous Stores,CITY_CENTROID,false
HIST_TX_3c28f05dd876e7631fb7e569aaccbd18288bacaf57836d61ad35790bcf68c80e,IBM_USER_134,IBM_CARD_134_4,62.49,USD,ONLINE,ECOMMERCE,INFERRED_FROM_METHOD,Business Services,UNKNOWN,false
HIST_TX_66132cc16e463295b822d7696731167f82b395c0c93dfb1f26206afd6e4b9d11,IBM_USER_387,IBM_CARD_387_0,1.79,USD,CHIP,ATM,PROJECT_ASSUMPTION,Retail Outlet Services,CITY_CENTROID,false
HIST_TX_a19a18e134376c79d7e7679e8c66e7ffd251081c55d5c66f5f33067bd7d63f44,IBM_USER_50,IBM_CARD_50_0,80.00,USD,SWIPE,POS,PROJECT_ASSUMPTION,Utility Services,UNKNOWN,false
HIST_TX_dec3d89b8edd09b33064ba5556d6981e6e053e3556b57cfacd7bc4aaa7e66dfe,IBM_USER_1710,IBM_CARD_1710_0,174.83,USD,SWIPE,POS,PROJECT_ASSUMPTION,Utility Services,UNKNOWN,false
HIST_TX_2cd2ba923bb712a7dc18d830d7d6e2ccf630a22954032d0e385d131465c81152,IBM_USER_956,IBM_CARD_956_0,58.00,USD,SWIPE,POS,PROJECT_ASSUMPTION,Retail Outlet Services,CITY_CENTROID,false
HIST_TX_6c9ccfe9b7e7047e7715b0098bb9ad69b60d35fd74dfbc165239f391b236d57f,IBM_USER_1940,IBM_CARD_1940_0,43.20,USD,ONLINE,ECOMMERCE,INFERRED_FROM_METHOD,Transportation Services,UNKNOWN,false
HIST_TX_8c0cadcbb62157c2e5161949b8e896933ecaf00969549c4a6334cbd3406ee9a0,IBM_USER_1644,IBM_CARD_1644_4,49.13,USD,SWIPE,POS,PROJECT_ASSUMPTION,Business Services,CITY_CENTROID,false


Ground truth remains separate: ['source_record_id', 'transaction_id', 'event_id', 'ground_truth_label', 'label_source']


In [0]:
# SIC8 Notebook 03 — Final Quality Gates (FULL snapshot only)
# Paste this as a NEW CELL immediately before the Delta Write section.
# Run the entire notebook from the top with ENABLE_WRITE = False first.

from pyspark.sql import functions as F
import time

audit_started = time.perf_counter()
FINAL_GATES_PASSED = False

if MODE != "FULL":
    raise RuntimeError("Final quality gates require MODE='FULL'.")
if ENABLE_WRITE:
    print("WRITE ENABLED: final gates must pass before the Delta Write section.")

# 1) Verify that Users indexing preserved every original CSV record.
user_fields = raw["users"].columns
original_users = raw["users"].select(*user_fields)
reconstructed_users = users_indexed.select(*user_fields)

missing_user_records = original_users.exceptAll(reconstructed_users).count()
extra_user_records = reconstructed_users.exceptAll(original_users).count()

print("Users missing/extra records:", missing_user_records, extra_user_records)
if missing_user_records or extra_user_records:
    raise AssertionError("Users reconstruction changed the CSV contents.")

# 2) Check transaction IDs across the entire historical source.
identity_stats = checked.agg(
    F.count("*").alias("rows"),
    F.count("transaction_id").alias("non_null_transaction_ids"),
    F.countDistinct("transaction_id").alias("unique_transaction_ids"),
    F.sum(F.xxhash64("transaction_id").cast("decimal(38,0)")).alias("id_checksum")
).first().asDict()

print("Transaction identity stats:", identity_stats)
if identity_stats["rows"] != source_count:
    raise AssertionError("ID audit row count differs from source count.")
if identity_stats["non_null_transaction_ids"] != source_count:
    raise AssertionError("Null transaction IDs detected.")
if identity_stats["unique_transaction_ids"] != source_count:
    raise AssertionError("Duplicate transaction IDs detected.")

# 3) Detect conflicting fraud labels among otherwise identical raw records.
# Notebook 03 orders duplicate occurrences by the source label, so
# conflicting labels need an identity-policy decision before publication.
conflicting_groups = (
    source_hashed
    .groupBy("source_row_hash")
    .agg(F.countDistinct(F.coalesce(F.col("Is Fraud?"), F.lit("<NULL>"))).alias("label_variants"))
    .filter(F.col("label_variants") > 1)
    .count()
)

print("Same-content groups with conflicting labels:", conflicting_groups)
if conflicting_groups:
    raise AssertionError(
        "Source identity depends on label ordering for mixed-label duplicate groups; "
        "review ID generation before publishing."
    )

# 4) Protect row counts and fraud-label isolation.
if summary["rows_after_joins"] != source_count:
    raise AssertionError("Enrichment changed transaction row count.")
if summary["accepted"] + summary["review"] != source_count:
    raise AssertionError("Accepted + review counts do not equal source rows.")
if "ground_truth_label" in silver_transactions.columns or "Is Fraud?" in silver_transactions.columns:
    raise AssertionError("Fraud label leaked into detector transaction columns.")
if "ground_truth_label" not in silver_ground_truth.columns:
    raise AssertionError("Ground-truth sidecar is missing its label.")

FINAL_GATES_PASSED = True
print("\nFINAL QUALITY GATES: PASS")
print("Source rows:", source_count)
print("Accepted/review:", summary["accepted"], summary["review"])
print("Identity checksum (record for second-run comparison):", identity_stats["id_checksum"])
print("Audit seconds:", round(time.perf_counter() - audit_started, 2))
print("Note: uniqueness + checksum do not prove an immutable physical CSV row ID.")


Users missing/extra records: 0 0
Transaction identity stats: {'rows': 24386900, 'non_null_transaction_ids': 24386900, 'unique_transaction_ids': 24386900, 'id_checksum': Decimal('1845018497720548795465')}
Same-content groups with conflicting labels: 0

FINAL QUALITY GATES: PASS
Source rows: 24386900
Accepted/review: 24386900 0
Identity checksum (record for second-run comparison): 1845018497720548795465
Audit seconds: 34.2
Note: uniqueness + checksum do not prove an immutable physical CSV row ID.


## 14 — Optional idempotent snapshot write (OFF by default)
Use managed Delta tables only after the full input and join gates pass. Snapshot `overwrite` avoids accumulating duplicates across reruns, but **the three tables are not committed atomically as one transaction**.

In [0]:

if ENABLE_WRITE and not globals().get("FINAL_GATES_PASSED", False):
    raise RuntimeError("Final quality gates must pass before writing")
if ENABLE_WRITE:
    if MODE != "FULL":
        raise ValueError("Never write production tables from SAMPLE mode")
    if summary["rows_after_joins"] != source_count:
        raise AssertionError("Row-count gate failed")

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

    outputs = {
        f"{SILVER_SCHEMA}.historical_transactions": silver_transactions,
        f"{SILVER_SCHEMA}.historical_ground_truth": silver_ground_truth,
        f"{SILVER_SCHEMA}.historical_quarantine": silver_quarantine,
    }
    for table_name, data in outputs.items():
        (data.write.format("delta").mode("overwrite")
            .option("overwriteSchema", "true").saveAsTable(table_name))
        print("Written:", table_name)
else:
    print("DRY RUN: no S3 or Delta writes performed.")
    print("To publish, inspect all checks, confirm writable managed storage, then ENABLE_WRITE=True.")

DRY RUN: no S3 or Delta writes performed.
To publish, inspect all checks, confirm writable managed storage, then ENABLE_WRITE=True.


## 15 — After-run verification / handoff
When writes are enabled, verify saved row counts and hand the downstream team table names plus a dated validation summary. The streaming CDC integration comes next; it is **not** part of this historical batch notebook.

In [0]:
if ENABLE_WRITE:
    saved_transactions = spark.table(f"{SILVER_SCHEMA}.historical_transactions").count()
    saved_labels = spark.table(f"{SILVER_SCHEMA}.historical_ground_truth").count()
    saved_quarantine = spark.table(f"{SILVER_SCHEMA}.historical_quarantine").count()
    assert saved_transactions + saved_quarantine == source_count
    assert saved_labels == saved_transactions
    print("Published counts:", saved_transactions, saved_labels, saved_quarantine)
else:
    print("Next: approve historical channel assumption, validate row IDs on full snapshot,")
    print("then enable Delta writes and verify counts before connecting streaming events.")

Next: approve historical channel assumption, validate row IDs on full snapshot,
then enable Delta writes and verify counts before connecting streaming events.


In [0]:

# Final quality gates before publishing Silver tables

import time

start = time.perf_counter()
FINAL_GATES_PASSED = False

if MODE != "FULL":
    raise ValueError("Final validation requires FULL mode")

# 1. Validate reconstructed Users against the original CSV
user_columns = raw["users"].columns

original_users = raw["users"].select(*user_columns)
reconstructed_users = users_indexed.select(*user_columns)

missing_users = original_users.exceptAll(reconstructed_users).count()
extra_users = reconstructed_users.exceptAll(original_users).count()

assert missing_users == 0, "Missing records in reconstructed Users"
assert extra_users == 0, "Unexpected records in reconstructed Users"

print("Users reconstruction: PASS")


# 2. Validate transaction identity across the full dataset
id_stats = checked.agg(
    F.count("*").alias("total_rows"),
    F.count("source_record_id").alias("non_null_ids"),
    F.countDistinct("source_record_id").alias("unique_ids"),
    F.sum(
        F.xxhash64("source_record_id").cast("decimal(38,0)")
    ).alias("identity_checksum")
).first().asDict()

assert id_stats["total_rows"] == source_count
assert id_stats["non_null_ids"] == source_count, "Null transaction IDs found"
assert id_stats["unique_ids"] == source_count, "Duplicate transaction IDs found"

print("Transaction identity: PASS")


# 3. Validate row reconciliation
assert summary["rows_after_joins"] == source_count
assert summary["accepted"] + summary["review"] == source_count

print("Row reconciliation: PASS")


# 4. Check fraud-label isolation
for column in ["Is Fraud?", "ground_truth_label"]:
    assert column not in silver_transactions.columns

assert "ground_truth_label" in silver_ground_truth.columns

print("Ground truth isolation: PASS")


# 5. Print validation summary
print("\nFINAL VALIDATION REPORT")
print("-" * 45)
print("Source rows:", source_count)
print("Unique IDs:", id_stats["unique_ids"])
print("Accepted:", summary["accepted"])
print("Review:", summary["review"])
print("Missing reconstructed users:", missing_users)
print("Extra reconstructed users:", extra_users)
print("Identity checksum:", id_stats["identity_checksum"])
print("Execution time:", round(time.perf_counter() - start, 2), "sec")

FINAL_GATES_PASSED = True
print("\nFINAL QUALITY GATES: PASSED")


Users reconstruction: PASS
Transaction identity: PASS
Row reconciliation: PASS
Ground truth isolation: PASS

FINAL VALIDATION REPORT
---------------------------------------------
Source rows: 24386900
Unique IDs: 24386900
Accepted: 24386900
Review: 0
Missing reconstructed users: 0
Extra reconstructed users: 0
Identity checksum: -4383976808919492862827
Execution time: 17.19 sec

FINAL QUALITY GATES: PASSED
